### Snake: Simulate a snake game and print the game results.

You are given a map that ‘x’ represents a rock, ‘-’represents a space, ‘#’represents the body of snake. ‘@’represent the head of the snake and a sequence of actions that ‘0,1,2,3’represent to move to up/down/left/right correspondingly for one step.
A greedy snake starts in the map state and moves one step per unit of time according to the sequence of actions until all actions complete or fail. It will fail when the head and the stone overlap, the head goes beyond the boundary, or the head overlaps the body. 

#### Input
A matrix with type char (the map). 
A sequence with type int (the motions). 

#### Output
the the result of the game:
If it failed, output the running time of the game.
It it didn’t fail, output the final position of the head (in the form “%d, %d”).

In [4]:
"""
Example:
input:
map:
---------
------x--
-x-------
---@-----
---##----
------x--
--x----x-
-x-------
---------
action:
0 0 3 3 0 3 3 1 1 1 1 1 3 1 1 2 2 2 2 2

output:
7 3
"""

'\nExample:\ninput:\nmap:\n---------\n------x--\n-x-------\n---@-----\n---##----\n------x--\n--x----x-\n-x-------\n---------\naction:\n0 0 3 3 0 3 3 1 1 1 1 1 3 1 1 2 2 2 2 2\n\noutput:\n7 3\n'

In [5]:
# add your code here

from collections import deque
# the difference between deque and queue is:
# deque is a double-ended queue, which allows adding and removing elements from both ends
# while queue is a single-ended queue, which only allows adding elements to the end and removing elements from the front

def snake_game(game_map, actions):
    rows = len(game_map) # get the number of rows
    cols = len(game_map[0]) # get the numbr of columns

    # define the direction vectors for up, down, left and right
    directions = [
        (-1, 0), # go up
        (1, 0), # go down
        (0, -1), # go left
        (0, 1) # go right
    ]

    # find the initial position of the head, based on '@' in map
    head = None
    for row in range(rows):
        for col in range(cols):
            if game_map[row][col] == '@':
                head = (row, col)
                break

        if head is not None:
            break

    # no head exists at all, return an error
    if head is None:
        raise ValueError("The snake's head '@' is missing in the game map")

    # initialize the snake as a deque, with the head at the front
    snake = deque([head])

    # reconstruct the whole body, starting from the head
    # the snake itself can be considered as a connected component of the map
    current = head # everytime we visit all four directions extended from the current position

    # maintain a set to keep track of all visited positions
    # so that we can avoid revisiting the same position again.
    visited = {head}

    # traverse through the snake's body, until we reach the tail
    # the tail has no adjacent body part around, except for the previous one, which is the second last body part
    # after the end condition is triggered, the snake deque will contain all grid cells of the snake's body, from head to tail
    while True:
        current_row, current_col = current
        next_body = None

        # find the next body part, which is a '#' adjacent to the current position
        # use directions to find all four adjacent positions (up, down, left, right)
        for d_row, d_col in directions:

            # get the next position:
            next_row, next_col = current_row + d_row, current_col + d_col
            next_position = (next_row, next_col)

            # check if the next position is within bounds
            if not (0 <= next_row < rows and 0 <= next_col < cols):
                continue

            # also check if the next position is a body part '#' and not visited before
            if (game_map[next_row][next_col] == '#' and next_position not in visited):
                next_body = next_position # found the concrete next body part
                break

        if next_body is None:
            break # no adjacent body part found, except for the previous one -> reach the tail -> end the while loop

        visited.add(next_body) # add to the visited set, to avoid revisiting the same position
        snake.append(next_body) # add to the deque, as part of the snake's body
        current = next_body # move to the next body part

    # time means the number of actions taken, one action -> one time unit
    # action is a number from 0 to 3, representing the direction of movement
    for time, action in enumerate(actions, start=1):
        # get the current position of the head of the snake
        # the head is always at the front of the deque
        current_row, current_col = snake[0]

        # get the direction vector, based on the action (a number)
        d_row, d_col = directions[action]

        # get next position of the head of the snake
        new_head = (
            current_row + d_row,
            current_col + d_col
        )
        new_row, new_col = new_head

        # check if the next position is out of bounds
        if not (0 <= new_row < rows and 0 <= new_col < cols):
            return time # snake hits the surrounding wall, game over

        # check if the next position is a rock or its body
        if game_map[new_row][new_col] in ('x', '#'):
            return time # snake hits a rock or its body, game over

        # update the position of head & body of the snake
        # since the snake is moving one step at a time, we can just update the head & tail of the snake
        # and the rest of the body will follow the head, so do not need to keep track of the whole body of the snake
        # actually we implement by updating the configuration of the map
        # e.g. ####@- (current)
        #      -####@ (next)
        # only three grid cells are changed:
        # 1. '-' -> '@'; 2. '@' -> '#'; 3. '#' -> '-'
        # the rest of the body remains the same -> no need to update them

        # update the deque:
        # append one in & pop one out -> length remains the same

        current_head = snake[0] # get a copy of the current head position

        # add the new head position to the front of the deque
        snake.appendleft(new_head)

        current_tail = snake.pop() # remove tail and get a copy of the current tail position

        # update the game map:
        game_map[new_row][new_col] = '@' # new head position
        game_map[current_head[0]][current_head[1]] = '#' # current head becomes body
        game_map[current_tail[0]][current_tail[1]] = '-' # current tail becomes empty

    final_row, final_col = snake[0] # get the final position of the head, at the end of all actions
    return f"{final_row} {final_col}" # follow the output format to print


In [6]:
# test block
test_case = 4

with open(f'test_cases/problem3/{test_case}-map.txt', 'r') as f:
    game_map = [list(line.strip()) for line in f]

with open(f'test_cases/problem3/{test_case}-actions.txt', 'r') as f:
    actions = list(map(int, f.read().split()))

result = snake_game(game_map, actions)

print(result)

33
